# Module 7: Deploy
> ~40 min · optional

In Module 3, Engine found why `get_client_profile` keeps failing. Here you ship the same agent to a Cloud **deployment** in its **Staging** environment, once with `langgraph deploy` and once as a **Managed Deep Agent**. Then you point Engine at the staging deployment, and it replays the failing threads there to prove the issue reproduces. With the deployment live, Module 3's **Red-team a deployment** section can run too.

> **Project-based or self-hosted workspace?** Replace `--agent-id {AGENT_NAME} --agent-environment {DEPLOY_ENVIRONMENT}` with `--name client-research`, and call `find_deployment(name="client-research")`; the deployment gets its own tracing project. Self-hosted also needs `--push-to` with your own registry, and fix validation is Cloud-only.

## Setup
This cell loads your keys and the agent's name and environment from `utils/config.py`, the same values every module uses.

In [ ]:
import os
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.config import AGENT_NAME, DEPLOY_ENVIRONMENT, MANAGED_AGENT_NAME
from utils.deployment import find_deployment, reply_text, write_env_subset
from utils.models import API_KEY_ENV

# The langgraph CLI reads the workspace from this name when the key is org-scoped.
if os.environ.get("LANGSMITH_WORKSPACE_ID"):
    os.environ.setdefault("LANGSMITH_TENANT_ID", os.environ["LANGSMITH_WORKSPACE_ID"])

print(f"Agent: {AGENT_NAME} → {DEPLOY_ENVIRONMENT}")
!cd "{project_root}" && uv run langgraph --version

> **Requires:** a LangSmith Cloud workspace with Deployments enabled, and a personal access token or service key that can create deployments (a **Tracing Only** key can't). **Without it:** read sections 1 and 2, then skip to the Recap.

## 1. The deployable project
A deployment is built from `langgraph.json` at the repo root, which names the graphs the server exposes. The `client_research` graph imports `get_client_profile` and `web_search` from `agents/client_research_agent.py`, the module your notebooks ran, so the deployment ships the behavior Engine flagged in Module 3.

In [ ]:
import json

langgraph_config = json.loads((project_root / "langgraph.json").read_text())
print("Graphs:", json.dumps(langgraph_config["graphs"], indent=2))

deployable_source = project_root / "agents/deployable_agents/client_research/agent.py"
for line in deployable_source.read_text().splitlines():
    if line.startswith("from agents.client_research_agent import"):
        print("Shared tools:", line)

!cd "{project_root}" && uv run langgraph validate

`langgraph validate` checks `langgraph.json` itself (graph paths, Python version, dependencies) without building anything, so a config mistake shows up here instead of five minutes into a build.

## 2. Deployment secrets
`langgraph deploy` uploads each entry of the env file that `langgraph.json` names as a **deployment secret**, except reserved names such as `LANGSMITH_API_KEY`, which the platform sets itself. The cell below writes `.env.deploy` with only the model key and the Tavily key, so tracing settings from your `.env` can't send staging traces to Development.

In [ ]:
deploy_secret_names = [API_KEY_ENV, "TAVILY_API_KEY"]
written = write_env_subset(
    source=project_root / ".env",
    target=project_root / ".env.deploy",
    keys=deploy_secret_names,
)
print("Deployment secrets:", ", ".join(written))

Expect `LANGSMITH_GATEWAY_API_KEY, TAVILY_API_KEY`: the deployed agent calls its model through the same LLM Gateway pair as `utils/models.py`, and its own LangSmith key comes from the platform. `.env.deploy` is gitignored, like `.env`.

## 3. Deploy to the staging environment
`langgraph deploy --agent-id --agent-environment` binds the deployment to your registered agent, so its traces land in **Staging** next to the **Development** traces from your notebooks. `--remote` builds on LangSmith's builder, so Docker isn't needed. The first deploy takes about 5–10 minutes; rerunning the cell pushes a new revision to the same deployment.

In [ ]:
!cd "{project_root}" && uv run langgraph deploy --agent-id {AGENT_NAME} --agent-environment {DEPLOY_ENVIRONMENT} --remote --no-input

In [ ]:
staging = find_deployment(agent_id=AGENT_NAME, environment=DEPLOY_ENVIRONMENT)
print("Deployment ID:", staging.id)
print("API URL:      ", staging.api_url)

Open your agent's **Overview**: the **Deployments** section has a new row in **Staging**. Click it to open the deployment page, which shows the same **API URL**. If the build failed, `langgraph deploy logs --deployment-id <Deployment ID> --type build` shows why.

<img src="../images/TODO-07-agent-deployments-row.png" alt="Agent Overview with a Staging deployment row" style="display: block; margin: 0; width: auto; max-height: 360px; border-radius: 8px;">

*Agent Overview → Deployments*

<img src="../images/deployment_home.png" alt="LangSmith deployment page" style="display: block; margin: 0; width: auto; max-height: 360px; border-radius: 8px;">

*The deployment page*

## 4. Call the deployment
The deployment is an **Agent Server**: the same graph behind an API with threads, persistence and an `/mcp` endpoint. The cell below sends three client-lookup questions, each on its own thread, and takes about a minute.

In [ ]:
from langgraph_sdk import get_sync_client

LOOKUP_QUESTIONS = [
    "Pull up Acme Pension Fund's current holdings.",
    "What's the risk profile for the Smith Family Office?",
    "Prep me for tomorrow's meeting with TechCorp Treasury.",
]

server = get_sync_client(url=staging.api_url, api_key=os.environ["LANGSMITH_API_KEY"])
for question in LOOKUP_QUESTIONS:
    thread = server.threads.create()
    result = server.runs.wait(
        thread["thread_id"],
        "client_research",
        input={"messages": [{"role": "user", "content": question}]},
    )
    print(f"Q: {question}\nA: {reply_text(result)[:200]}\n")

print("API docs:", f"{staging.api_url}/docs")

Open **Tracing**, switch the environment to **Staging**, and open one of these threads: most show the same failing `get_client_profile` calls you saw in Development, because the deployment runs the same code. The **API docs** link opens the server's reference for every endpoint.

<img src="../images/deployment_docs.png" alt="Agent Server API reference" style="display: block; margin: 0; width: auto; max-height: 360px; border-radius: 8px;">

*`/docs` on the deployment*

## 5. The same agent as a Managed Deep Agent
A **Managed Deep Agent** is a folder (`agent.py`, `instructions.md`, `skills/`) that LangSmith compiles and hosts for you, with no `langgraph.json`, backend or checkpointer to write. `agents/managed/client_research/` holds the research half of the agent: the same skills, Tavily search, and no client lookup. `mda deploy` builds it remotely in about 5 minutes and registers it as its own agent, `client-research-managed`.

> **Requires:** Managed Deep Agents (public beta, LangSmith Cloud) and the `mda` CLI (`uv tool install managed-deepagents`). **Without it:** skip to section 6.

In [ ]:
mda_dir = project_root / "agents/managed/client_research"
mda_secrets = write_env_subset(
    source=project_root / ".env",
    target=mda_dir / ".env",
    keys=deploy_secret_names,
)
print("Managed agent secrets:", ", ".join(mda_secrets))

# mda forwards the folder's .env entries as deployment secrets.
!cd "{mda_dir}" && mda deploy .

`mda deploy` also copied `instructions.md` and `skills/` into **Context Hub**. Open the `client-research-managed` deployment, follow the **Context Hub** link in its **Details** sidebar, add the line "End every answer with a one-line summary that starts with TL;DR." to `instructions.md`, and save. Then run the next cell.

<img src="../images/TODO-07-mda-context-hub.png" alt="Context Hub editing instructions.md" style="display: block; margin: 0; width: auto; max-height: 360px; border-radius: 8px;">

*Context Hub → instructions.md*

In [ ]:
RESEARCH_QUESTION = "What moved Microsoft's stock this week? Search at most twice."

managed = find_deployment(name=MANAGED_AGENT_NAME)
managed_server = get_sync_client(url=managed.api_url, api_key=os.environ["LANGSMITH_API_KEY"])
thread = managed_server.threads.create()
result = managed_server.runs.wait(
    thread["thread_id"],
    MANAGED_AGENT_NAME,
    input={"messages": [{"role": "user", "content": RESEARCH_QUESTION}]},
)
print("Answer (last 400 chars):", reply_text(result)[-400:])

The answer ends with your TL;DR line, with no rebuild: Context Hub holds the instructions and skills, and the deployment holds the code.

> A later `mda deploy` syncs the folder's `instructions.md` to Context Hub again, so copy your edit into the file first if you want to keep it.

## 6. Reproduce the Engine issue on the deployment
**Fix validation** replays the threads behind an Engine issue against a **baseline deployment** and checks whether the issue happens again. Open **Engine**, select your agent's **Development** environment, click **Configure Engine**, paste the **Deployment ID** from section 3 into **Deployment** → **Baseline deployment**, and click **Save**. Then open the `get_client_profile` issue from Module 3 and click **Test issue**.

> **Requires:** fix validation (private beta, LangSmith Cloud) and `deployments:update` on the deployment. **Without it:** open the issue's **Evidence**, copy one thread's first message, and send it with the section 4 cell.

<img src="../images/TODO-07-engine-test-issue.png" alt="Engine issue after Test issue" style="display: block; margin: 0; width: auto; max-height: 360px; border-radius: 8px;">

*Engine → issue → Test issue*

Engine replays up to five of the issue's threads, judges each replay **Reproduced**, **Not reproduced** or **Inconclusive**, and records them as an experiment you can compare a fixed revision against later. The issue moves from **Awaiting test** to **Reproduced**; if no replay reproduces it, Engine closes the issue. If you forked the repo and opened Engine's PR in Module 3, Engine can also verify that fix on a **preview deployment**, which needs a deployment created from GitHub with preview builds triggered by the `preview` label (see **Docs**).

## Recap

| Step | Where it lives |
|---|---|
| Describe what ships | `langgraph.json`, `langgraph validate` |
| Choose deployment secrets | `.env.deploy`, named by `langgraph.json` `env` |
| Deploy to Staging | `langgraph deploy --agent-id --agent-environment --remote` |
| Watch it | agent **Overview** → **Deployments**; **Tracing** → **Staging** |
| Call it | `langgraph_sdk` `threads.create`, `runs.wait`; `/docs`, `/mcp` |
| Host a folder | `mda deploy`; **Context Hub** |
| Prove the issue | **Configure Engine** → **Baseline deployment**; **Test issue** |

**Next:** you've finished the tour. Module 3's **Red-team a deployment** section can now probe this deployment.

**Docs:** [Deploy to an agent environment](https://docs.langchain.com/langsmith/deploy-to-agent-environment) · [LangGraph CLI](https://docs.langchain.com/langsmith/cli) · [Agent Server](https://docs.langchain.com/langsmith/agent-server-overview) · [MCP endpoint](https://docs.langchain.com/langsmith/server-mcp) · [Managed Deep Agents](https://docs.langchain.com/langsmith/managed-deep-agents-overview) · [Context Hub for MDA](https://docs.langchain.com/langsmith/managed-deep-agents-context-hub) · [Validate fixes](https://docs.langchain.com/langsmith/engine-fix-validation) · [Preview builds](https://docs.langchain.com/langsmith/preview-builds)